In [ ]:
RESIDUE_MASS = {
    "A": 71.0788,  "R": 156.1875, "N": 114.1038, "D": 115.0886,
    "C": 103.1388, "E": 129.1155, "Q": 128.1307, "G": 57.0519,
    "H": 137.1411, "I": 113.1594, "L": 113.1594, "K": 128.1741,
    "M": 131.1926, "F": 147.1766, "P": 97.1167,  "S": 87.0782,
    "T": 101.1051, "W": 186.2132, "Y": 163.1760, "V": 99.1326,
}

WATER_MASS = 18.01524
N_TERM_PKA = 9.093
C_TERM_PKA = 2.340

SIDE_CHAIN_PKA = {
    "D": (3.65, "negative"),
    "E": (4.25, "negative"),
    "C": (8.30, "negative"),
    "Y": (10.07, "negative"),
    "H": (6.00, "positive"),
    "K": (10.53, "positive"),
    "R": (12.48, "positive"),
}


def _clean(seq: str) -> str:
    return seq.strip().upper()

def validate_protein(seq: str) -> bool:
    return set(_clean(seq)) <= set(RESIDUE_MASS.keys())


def molecular_weight(seq: str) -> float:
    seq = _clean(seq)
    if not seq:
        raise ValueError("Sequence is empty.")
    if not validate_protein(seq):
        bad = sorted(set(seq) - set(RESIDUE_MASS.keys()))
        raise ValueError(f"Unrecognized amino acid code(s): {bad}")

    return sum(RESIDUE_MASS[aa] for aa in seq) + WATER_MASS


def _net_charge_at_ph(seq: str, ph: float) -> float:
    charge = 1.0 / (1.0 + 10 ** (ph - N_TERM_PKA))
    charge -= 1.0 / (1.0 + 10 ** (C_TERM_PKA - ph))

    for aa in seq:
        if aa in SIDE_CHAIN_PKA:
            pka, kind = SIDE_CHAIN_PKA[aa]
            if kind == "positive":
                charge += 1.0 / (1.0 + 10 ** (ph - pka))
            else:
                charge -= 1.0 / (1.0 + 10 ** (pka - ph))
    return charge


def calculate_pI(seq: str, tolerance: float = 1e-4) -> float:
    seq = _clean(seq)
    if not seq:
        raise ValueError("Sequence is empty.")
    if not validate_protein(seq):
        bad = sorted(set(seq) - set(RESIDUE_MASS.keys()))
        raise ValueError(f"Unrecognized amino acid code(s): {bad}")

    lo, hi = 0.0, 14.0
    charge_lo = _net_charge_at_ph(seq, lo)
    charge_hi = _net_charge_at_ph(seq, hi)

    if charge_lo * charge_hi > 0:
        raise RuntimeError("Could not bracket a pI root between pH 0 and 14.")

    while hi - lo > tolerance:
        mid = (lo + hi) / 2
        charge_mid = _net_charge_at_ph(seq, mid)
        if charge_mid > 0:
            lo = mid
        else:
            hi = mid

    return (lo + hi) / 2


def amino_acid_composition(seq: str) -> dict:
    seq = _clean(seq)
    composition = {}
    for aa in seq:
        composition[aa] = composition.get(aa, 0) + 1
    return composition


if __name__ == "__main__":
    sample_protein = input("Enter a protein sequence: ")

    print(f"Sample protein sequence : {sample_protein}")
    print(f"Length                  : {len(sample_protein)} residues")
    print(f"Molecular weight        : {molecular_weight(sample_protein):.2f} Da")
    print(f"Isoelectric point (pI)  : {calculate_pI(sample_protein):.2f}")

    print("\nAmino acid composition:")
    for aa, count in sorted(amino_acid_composition(sample_protein).items()):
        print(f"  {aa}: {count}")